In [5]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, to_date, trim,round
from pyspark.sql.types import DoubleType, IntegerType

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [6]:
spark = SparkSession.builder \
.master("local") \
.appName("Word Count") \
.config("spark.jars.packages", "com.crealytics:spark-excel_2.11:0.13.1") \
.getOrCreate()


VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [3]:
sales_df = spark.read.csv("s3://uberads-analysis-us-west-2-tmp90days/ishika/demo/ac_data/sales/*", header=True, inferSchema=True)    
# sales_df.show(5)


# Cleaning

clean_sales_df = (
    sales_df.withColumn("sale_date", to_date(col("sale_date"), "yyyy-MM-dd"))
    .withColumn("product_id", trim(col("product_id")))
    .withColumn("customer_id", trim(col("customer_id")))
    .withColumn("quantity", col("quantity").cast(IntegerType()))
    .withColumn("product_price", col("product_price").cast(DoubleType()))
    .withColumn("total_sale_amount", col("total_sale_amount").cast(DoubleType()))
    .dropna(subset=["sale_date", "product_id", "customer_id", "quantity", "product_price","total_sale_amount"])
)
# clean_sales_df.show(5)

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

+---------+----------+----------+-----------+--------+-------------+-----------------+
|  sale_id| sale_date|product_id|customer_id|quantity|product_price|total_sale_amount|
+---------+----------+----------+-----------+--------+-------------+-----------------+
|S06466700|2023-04-08|   PID1005|      C1354|     6.0|        38.09|           228.54|
|S06466701|2023-04-08|   PID1035|      C1105|    17.0|        42.07|           715.19|
|S06466702|2023-04-08|   PID1067|      C1274|    20.0|        60.96|           1219.2|
|S06466703|2023-04-08|   PID1004|      C1340|    14.0|        51.25|            717.5|
|S06466704|2023-04-08|   PID1065|      C1488|    20.0|        25.45|            509.0|
+---------+----------+----------+-----------+--------+-------------+-----------------+
only showing top 5 rows

+---------+----------+----------+-----------+--------+-------------+-----------------+
|  sale_id| sale_date|product_id|customer_id|quantity|product_price|total_sale_amount|
+---------+-------

In [15]:
# Read data from S3

products_df = spark.read.csv(
    "s3://uberads-analysis-us-west-2-tmp90days/ishika/demo/ac_data/products.csv",
    header=True,
    inferSchema=True
)

inventory_df = spark.read.csv(
    "s3://uberads-analysis-us-west-2-tmp90days/ishika/demo/ac_data/inventory.csv",
    header=True,
    inferSchema=True
)

# ---------------- Inventory Cleanup ----------------


inventory_df = (
    inventory_df.withColumn("warehouse_id", trim(col("warehouse_id")))
                .withColumn("product_id", trim(col("product_id")))
                .withColumn("stock_level", col("stock_level").cast(IntegerType()))
                .withColumn("reorder_level", col("reorder_level").cast(IntegerType()))
                .withColumn("avg_daily_sales", col("avg_daily_sales").cast(DoubleType()))
                .withColumn("days_until_reorder", round(col("days_until_reorder").cast(DoubleType()), 2))
                .dropna(subset=[
                    "warehouse_id", "product_id", "stock_level",
                    "reorder_level", "avg_daily_sales", "days_until_reorder"
                ])
                .filter(
                    (col("stock_level") >= 0) &
                    (col("reorder_level") >= 0) &
                    (col("avg_daily_sales") >= 0)
                )
)


# ---------------- Products Cleanup ----------------


products_df = (
    products_df.withColumn("product_id", trim(col("product_id")))
               .withColumn("product_name", trim(col("product_name")))
               .withColumn("category", trim(col("category")))
               .withColumn("price", col("price").cast(DoubleType()))
               .dropna(subset=["product_id", "product_name", "category", "price"])
               .filter(col("price") > 0)
)

# inventory_df.show(5)
# products_df.show(5)

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

+------------+----------+-----------+-------------+---------------+------------------+
|warehouse_id|product_id|stock_level|reorder_level|avg_daily_sales|days_until_reorder|
+------------+----------+-----------+-------------+---------------+------------------+
|    W000_000|   PID1000|         67|           89|           51.0|               1.3|
|    W000_001|   PID1001|         56|           63|           30.0|               1.9|
|    W000_002|   PID1002|        137|           87|           86.0|               1.6|
|    W000_003|   PID1003|         24|           68|           16.0|               1.5|
|    W000_004|   PID1004|         65|           84|           35.0|               1.9|
+------------+----------+-----------+-------------+---------------+------------------+
only showing top 5 rows

+----------+------------+----------+-----+
|product_id|product_name|  category|price|
+----------+------------+----------+-----+
|   PID1000|      Drug_0| Antiviral| 7.31|
|   PID1001|      Dr

In [22]:
#transformation

# Join with products and customers
customers_df = spark.read.option("multiline", "true").json(
    "s3://uberads-analysis-us-west-2-tmp90days/ishika/demo/ac_data/customers.json"
)
# customers_df.show()


# Clean joins

joined_df = clean_sales_df.join(products_df, on="product_id", how="left") \
                        .join(customers_df, on="customer_id", how="left")

# Add total_amount column

final_df = joined_df.withColumn(
    "total_amount",
    round(col("price") * col("quantity"), 2)
)


VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

+-----------+-----------+--------------------+---------+---------+---------+
|       city|customer_id|       customer_name|   region|sales_rep|     type|
+-----------+-----------+--------------------+---------+---------+---------+
|    Phoenix|      C1000|        Jeremy Brown|    Wesst|    Rep_1|   Retail|
|      Basel|      C1001|          Tony Meyer|     West|    Rep_3|   Online|
|Los Angeles|      C1002|  Christopher Sexton|    North|    Rep_4|   Retail|
|    Houston|      C1003|      Victoria Gross|  Central|    Rep_5|   Retail|
|      Delhi|      C1004|     Martha Mckenzie|  Central|    Rep_7|Wholesale|
|    Phoenix|      C1005|       Sarah Whitney|     East|    Rep_2|   Retail|
|Los Angeles|      C1006|        Ryan Johnson|   Cental|    Rep_5|Wholesale|
|    Phoenix|      C1007|       James Herrera|Southwest|   Rep_10|Wholesale|
|   New York|      C1008|        Jerry Benson|Southwest|    Rep_6|Wholesale|
|    Houston|      C1009|        Gary Carlson|Northeast|    Rep_8| Hospital|

In [24]:
#load
# final_df.show()
final_df.write.mode("overwrite").parquet("s3://uberads-analysis-us-west-2-tmp90days/ishika/demo/ac_data/final_sales/")


VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…